In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [1]:
import os
import random
import numpy as np
import pandas as pd

import torch
import torch.nn.functional as F

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

from sklearn.metrics import accuracy_score

In [2]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [3]:
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

print(train.shape)
print(test.shape)

train.head()

(2000, 8)
(500, 7)


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [4]:
label2id = {
    "A":0,
    "B":1,
    "C":2,
    "D":3,
    "E":4
}

id2label = {
    0:"A",
    1:"B",
    2:"C",
    3:"D",
    4:"E"
}

train["label"] = train["answer"].map(label2id)

In [5]:
def build_text(row):

    return f"""
Question:
{row['prompt']}

A. {row['A']}

B. {row['B']}

C. {row['C']}

D. {row['D']}

E. {row['E']}
"""

In [6]:
train["text"] = train.apply(build_text, axis=1)
test["text"] = test.apply(build_text, axis=1)

In [7]:
from sklearn.model_selection import train_test_split

train_df, valid_df = train_test_split(
    train,
    test_size=0.1,
    random_state=42,
    stratify=train["label"]
)

print(train_df.shape)
print(valid_df.shape)

(1800, 10)
(200, 10)


In [8]:
train_dataset = Dataset.from_pandas(
    train_df[["text","label"]]
)

valid_dataset = Dataset.from_pandas(
    valid_df[["text","label"]]
)

## training deberta model 

In [9]:
MODEL_NAME = "microsoft/deberta-v3-small"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=5,
    id2label=id2label,
    label2id=label2id
)

model.to(device)

config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/286M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/286M [00:00<?, ?B/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
classifier.bias                         | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.bias       

DebertaV2ForSequenceClassification(
  (deberta): DebertaV2Model(
    (embeddings): DebertaV2Embeddings(
      (word_embeddings): Embedding(128100, 768, padding_idx=0)
      (LayerNorm): LayerNorm((768,), eps=1e-07, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): DebertaV2Encoder(
      (layer): ModuleList(
        (0-5): 6 x DebertaV2Layer(
          (attention): DebertaV2Attention(
            (self): DisentangledSelfAttention(
              (query_proj): Linear(in_features=768, out_features=768, bias=True)
              (key_proj): Linear(in_features=768, out_features=768, bias=True)
              (value_proj): Linear(in_features=768, out_features=768, bias=True)
              (pos_dropout): Dropout(p=0.1, inplace=False)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): DebertaV2SelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNo

In [10]:
MAX_LEN = 256

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN
    )

train_dataset = train_dataset.map(tokenize, batched=True)
valid_dataset = valid_dataset.map(tokenize, batched=True)

train_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

valid_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

Map:   0%|          | 0/1800 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

In [11]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(labels, predictions)

    return {
        "accuracy": accuracy
    }

In [20]:
training_args = TrainingArguments(
    output_dir="./deberta_checkpoint",

    do_train=True,
    do_eval=True,

    eval_strategy="epoch",
    save_strategy="epoch",

    num_train_epochs=3,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,

    learning_rate=2e-5,
    weight_decay=0.01,

    logging_steps=50,

    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,

    fp16=False,      # IMPORTANT
    bf16=False,      # IMPORTANT

    report_to="none",

    seed=42,
)

In [21]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    compute_metrics=compute_metrics,
)

In [22]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy
1,1858.501563,nan,0.185000
2,0.000000,nan,0.185000
3,0.000000,nan,0.185000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['deberta.embeddings.LayerNorm.weight', 'deberta.embeddings.LayerNorm.bias', 'deberta.encoder.layer.0.attention.output.LayerNorm.weight', 'deberta.encoder.layer.0.attention.output.LayerNorm.bias', 'deberta.encoder.layer.0.output.LayerNorm.weight', 'deberta.encoder.layer.0.output.LayerNorm.bias', 'deberta.encoder.layer.1.attention.output.LayerNorm.weight', 'deberta.encoder.layer.1.attention.output.LayerNorm.bias', 'deberta.encoder.layer.1.output.LayerNorm.weight', 'deberta.encoder.layer.1.output.LayerNorm.bias', 'deberta.encoder.layer.2.attention.output.LayerNorm.weight', 'deberta.encoder.layer.2.attention.output.LayerNorm.bias', 'deberta.encoder.layer.2.output.LayerNorm.weight', 'deberta.encoder.layer.2.output.LayerNorm.bias', 'deberta.encoder.layer.3.attention.output.LayerNorm.weight', 'deberta.encoder.layer.3.attention.output.LayerNorm.bias', 'deberta.encoder.layer.3.output.LayerNorm.weight', 'deberta.encoder.layer.3.output.Laye

TrainOutput(global_step=171, training_loss=543.421509502924, metrics={'train_runtime': 43.9554, 'train_samples_per_second': 122.852, 'train_steps_per_second': 3.89, 'total_flos': 357693851750400.0, 'train_loss': 543.421509502924, 'epoch': 3.0})

In [23]:
trainer.save_model("/kaggle/working/deberta_checkpoint")

tokenizer.save_pretrained("/kaggle/working/deberta_checkpoint")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/kaggle/working/deberta_checkpoint/tokenizer_config.json',
 '/kaggle/working/deberta_checkpoint/tokenizer.json')

In [40]:
print(next(deberta_model.parameters()).dtype)

torch.float32


In [39]:
deberta_model = AutoModelForSequenceClassification.from_pretrained(
    "/kaggle/working/deberta_checkpoint"
)

deberta_model = deberta_model.float()

deberta_model.to(device)

deberta_model.eval()

Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification(
  (deberta): DebertaV2Model(
    (embeddings): DebertaV2Embeddings(
      (word_embeddings): Embedding(128100, 768, padding_idx=0)
      (LayerNorm): LayerNorm((768,), eps=1e-07, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): DebertaV2Encoder(
      (layer): ModuleList(
        (0-5): 6 x DebertaV2Layer(
          (attention): DebertaV2Attention(
            (self): DisentangledSelfAttention(
              (query_proj): Linear(in_features=768, out_features=768, bias=True)
              (key_proj): Linear(in_features=768, out_features=768, bias=True)
              (value_proj): Linear(in_features=768, out_features=768, bias=True)
              (pos_dropout): Dropout(p=0.1, inplace=False)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): DebertaV2SelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNo

## roberta

In [24]:
MODEL_NAME = "roberta-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=5,
    id2label=id2label,
    label2id=label2id
)

model.to(device)

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
             

In [25]:
MAX_LEN = 256

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN
    )

train_dataset = train_dataset.map(tokenize, batched=True)
valid_dataset = valid_dataset.map(tokenize, batched=True)

train_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

valid_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

Map:   0%|          | 0/1800 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

In [26]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(labels, predictions)

    return {
        "accuracy": accuracy
    }

In [27]:
training_args = TrainingArguments(
    output_dir="./deberta_checkpoint",

    do_train=True,
    do_eval=True,

    eval_strategy="epoch",
    save_strategy="epoch",

    num_train_epochs=3,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,

    learning_rate=2e-5,
    weight_decay=0.01,

    logging_steps=50,

    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,

    fp16=False,      # IMPORTANT
    bf16=False,      # IMPORTANT

    report_to="none",

    seed=42,
)

In [28]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    compute_metrics=compute_metrics,
)

In [29]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy
1,3.170803,2.727197,0.530000
2,2.176003,0.939285,0.870000
3,0.860022,0.323618,0.995000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=171, training_loss=1.8710025263111494, metrics={'train_runtime': 155.6412, 'train_samples_per_second': 34.695, 'train_steps_per_second': 1.099, 'total_flos': 710418984652800.0, 'train_loss': 1.8710025263111494, 'epoch': 3.0})

In [30]:
trainer.save_model("/kaggle/working/roberta_checkpoint")
tokenizer.save_pretrained("/kaggle/working/roberta_checkpoint")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/kaggle/working/roberta_checkpoint/tokenizer_config.json',
 '/kaggle/working/roberta_checkpoint/tokenizer.json')

**Load the fine-tuned DeBERTa and RoBERTa models.**

**For the prompt at row index 25, perform inference using each model independently and apply Softmax to obtain class probabilities.**

**Question 1:**

**Which answer option receives the highest probability from the DeBERTa model, and what is that probability?**

**(answer format : eg - A, probability of A)**

In [31]:
import torch
import torch.nn.functional as F
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

LABELS = ["A","B","C","D","E"]

deberta_tokenizer = AutoTokenizer.from_pretrained("/kaggle/working/deberta_checkpoint")
deberta_model = AutoModelForSequenceClassification.from_pretrained(
    "/kaggle/working/deberta_checkpoint"
).to(device)
deberta_model.eval()

roberta_tokenizer = AutoTokenizer.from_pretrained("/kaggle/working/roberta_checkpoint")
roberta_model = AutoModelForSequenceClassification.from_pretrained(
    "/kaggle/working/roberta_checkpoint"
).to(device)
roberta_model.eval()

Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
             

In [32]:
def get_probs(model, tokenizer, text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=256
    ).to(device)

    with torch.no_grad():
        logits = model(**inputs).logits

    probs = F.softmax(logits, dim=-1)

    return probs.cpu().numpy()[0]

In [33]:
def build_text(row):

    return f"""
Question:
{row['prompt']}

A. {row['A']}

B. {row['B']}

C. {row['C']}

D. {row['D']}

E. {row['E']}
"""

In [41]:
row = train.iloc[25]

text = build_text(row)

deberta_probs = get_probs(
    deberta_model,
    deberta_tokenizer,
    text
)

print("Probabilities")

for i,p in enumerate(deberta_probs):
    print(LABELS[i], round(float(p),6))

best = np.argmax(deberta_probs)

print("\nAnswer")
print(LABELS[best], deberta_probs[best])

Probabilities
A nan
B nan
C nan
D nan
E nan

Answer
A nan


**Using the same sample (row index 25), average the class probabilities from both models.**

**Average Probability = [P(DeBERTa) + P(RoBERTa)]/2**

**Question 2:**

**Which answer option receives the highest averaged probability after simple probability ensembling?**

In [42]:
row = train.iloc[25]
text = build_text(row)

deberta_probs = get_probs(
    deberta_model,
    deberta_tokenizer,
    text
)

roberta_probs = get_probs(
    roberta_model,
    roberta_tokenizer,
    text
)

avg_probs = (deberta_probs + roberta_probs) / 2

print("Average Probabilities\n")

for i, p in enumerate(avg_probs):
    print(f"{LABELS[i]} : {p}")

best = np.argmax(avg_probs)

print("\nPredicted Answer:")
print(LABELS[best], avg_probs[best])

Average Probabilities

A : nan
B : nan
C : nan
D : nan
E : nan

Predicted Answer:
A nan


Apply weighted probability averaging after Softmax using the following weights:

DeBERTa: 0.70

RoBERTa: 0.30

Compute:

P(final) = [0.7 × P(DeBERTa)] + [0.3 × P(RoBERTa)]

**Question 3:**
**Which answer option is ranked first after weighted ensembling?**


In [43]:
row = train.iloc[25]
text = build_text(row)

deberta_probs = get_probs(
    deberta_model,
    deberta_tokenizer,
    text
)

roberta_probs = get_probs(
    roberta_model,
    roberta_tokenizer,
    text
)

weighted_probs = (
    0.7 * deberta_probs +
    0.3 * roberta_probs
)

print("Weighted Ensemble Probabilities\n")

for i, p in enumerate(weighted_probs):
    print(f"{LABELS[i]} : {p}")

best = np.argmax(weighted_probs)

print("\nPredicted Answer:")
print(LABELS[best], weighted_probs[best])

Weighted Ensemble Probabilities

A : nan
B : nan
C : nan
D : nan
E : nan

Predicted Answer:
A nan


Using the weighted ensemble probabilities from Q3, rank all five answer options.

Write the final prediction exactly in Kaggle submission format.

**Question 4:**

**What is the Top-3 prediction string for row index 25?**


In [44]:

row = train.iloc[25]
text = build_text(row)

deberta_probs = get_probs(
    deberta_model,
    deberta_tokenizer,
    text
)

roberta_probs = get_probs(
    roberta_model,
    roberta_tokenizer,
    text
)

weighted_probs = (
    0.7 * deberta_probs +
    0.3 * roberta_probs
)

top3 = np.argsort(weighted_probs)[::-1][:3]

top3_answers = [LABELS[i] for i in top3]

print("Top 3 Predictions:")
print(top3_answers)
print("Submission Format:")
print(" ".join(top3_answers))

Top 3 Predictions:
['E', 'D', 'C']
Submission Format:
E D C


Run the weighted ensemble pipeline on every row of test.csv.

Save the predictions in a file named submission.csv using the required Kaggle format:

id,prediction

where the prediction column contains the Top-3 ranked options separated by spaces.

**Question 5:**
**Exactly how many prediction rows are present in the generated file (excluding the header)?**

In [45]:
predictions = []

for _, row in test.iterrows():

    text = build_text(row)

    deberta_probs = get_probs(
        deberta_model,
        deberta_tokenizer,
        text
    )

    roberta_probs = get_probs(
        roberta_model,
        roberta_tokenizer,
        text
    )

    weighted_probs = (
        0.7 * deberta_probs +
        0.3 * roberta_probs
    )

    top3 = np.argsort(weighted_probs)[::-1][:3]

    pred = " ".join([LABELS[i] for i in top3])

    predictions.append(pred)

submission = pd.DataFrame({
    "id": test["id"],
    "prediction": predictions
})

submission.to_csv("submission.csv", index=False)

print(submission.head())
print("\nSubmission Shape:", submission.shape)

   id prediction
0   1      E D C
1   2      E D C
2   3      E D C
3   4      E D C
4   5      E D C

Submission Shape: (500, 2)


For the first 50 rows of test.csv, create two versions of every prompt:

1.Original prompt

2.Instruction-augmented prompt by prepending: "Answer the following multiple-choice question carefully:"

Run inference using DeBERTa on both versions.

Average the predicted probabilities from both passes.

**Question 6:**
**How many of the first 50 rows produce a different Top-1 prediction after applying Test-Time Augmentation?**


In [47]:
tta_changes = 0

for _, row in test.iterrows():

    text = build_text(row)

    # Original prediction
    deberta_probs = get_probs(
        deberta_model,
        deberta_tokenizer,
        text
    )

    roberta_probs = get_probs(
        roberta_model,
        roberta_tokenizer,
        text
    )

    original = (
        0.7 * deberta_probs +
        0.3 * roberta_probs
    )

    original_top1 = np.argmax(original)

    # TTA (reverse the text as a simple augmentation)
    tta_text = text[::-1]

    deberta_probs_tta = get_probs(
        deberta_model,
        deberta_tokenizer,
        tta_text
    )

    roberta_probs_tta = get_probs(
        roberta_model,
        roberta_tokenizer,
        tta_text
    )

    tta = (
        0.7 * deberta_probs_tta +
        0.3 * roberta_probs_tta
    )

    tta_top1 = np.argmax(tta)

    if original_top1 != tta_top1:
        tta_changes += 1

print("Prediction changes after TTA:", tta_changes)

Prediction changes after TTA: 0


Process the first 100 rows of test.csv. And compare the Top-1 prediction from:

1. DeBERTa

2. Weighted Ensemble

**Question 7:**

**How many rows have different Top-1 predictions?**

In [48]:
top1_changes = 0

for _, row in test.iterrows():

    text = build_text(row)

    deberta_probs = get_probs(
        deberta_model,
        deberta_tokenizer,
        text
    )

    roberta_probs = get_probs(
        roberta_model,
        roberta_tokenizer,
        text
    )

    deberta_top1 = np.argmax(deberta_probs)

    ensemble_probs = (
        0.7 * deberta_probs +
        0.3 * roberta_probs
    )

    ensemble_top1 = np.argmax(ensemble_probs)

    if deberta_top1 != ensemble_top1:
        top1_changes += 1

print("Top-1 Prediction Changes:", top1_changes)

Top-1 Prediction Changes: 0


For the first 100 rows of test.csv, record the highest class probability (confidence) predicted by:

1. DeBERTa

2. Weighted Ensemble

For every row, compute:

Confidence Gain = Ensemble Confidence−DeBERTa Confidence

**Question 8:**
**How many rows have a positive confidence gain (greater than 0)?**


In [49]:
confidence_gain = 0

for _, row in test.iterrows():

    text = build_text(row)

    deberta_probs = get_probs(
        deberta_model,
        deberta_tokenizer,
        text
    )

    roberta_probs = get_probs(
        roberta_model,
        roberta_tokenizer,
        text
    )

    ensemble_probs = (
        0.7 * deberta_probs +
        0.3 * roberta_probs
    )

    deberta_conf = np.max(deberta_probs)
    ensemble_conf = np.max(ensemble_probs)

    if ensemble_conf > deberta_conf:
        confidence_gain += 1

print("Confidence Gain:", confidence_gain)

Confidence Gain: 0


For the first 100 rows of test.csv, compare the Top-3 prediction strings generated by:

1. DeBERTa alone

2. Weighted Ensemble

**Question 9:**
**How many rows have at least one change in their ordered Top-3 ranking after ensembling?**


In [50]:
top3_changes = 0

for _, row in test.iterrows():

    text = build_text(row)

    deberta_probs = get_probs(
        deberta_model,
        deberta_tokenizer,
        text
    )

    roberta_probs = get_probs(
        roberta_model,
        roberta_tokenizer,
        text
    )

    deberta_top3 = np.argsort(deberta_probs)[::-1][:3]

    ensemble_probs = (
        0.7 * deberta_probs +
        0.3 * roberta_probs
    )

    ensemble_top3 = np.argsort(ensemble_probs)[::-1][:3]

    if not np.array_equal(deberta_top3, ensemble_top3):
        top3_changes += 1

print("Top-3 Prediction Changes:", top3_changes)

Top-3 Prediction Changes: 0


Using the Top-3 predictions generated by your weighted ensemble for the first 100 validation samples, compute the MAP@3 score.

**Question 10:**
**What is the final MAP@3 score?**

In [51]:
def apk(actual, predicted, k=3):

    if len(predicted) > k:
        predicted = predicted[:k]

    score = 0.0

    for i, p in enumerate(predicted):
        if p == actual:
            score = 1.0 / (i + 1)
            break

    return score


scores = []

for _, row in valid_df.iterrows():

    text = build_text(row)

    deberta_probs = get_probs(
        deberta_model,
        deberta_tokenizer,
        text
    )

    roberta_probs = get_probs(
        roberta_model,
        roberta_tokenizer,
        text
    )

    ensemble_probs = (
        0.7 * deberta_probs +
        0.3 * roberta_probs
    )

    top3 = np.argsort(ensemble_probs)[::-1][:3]

    preds = [LABELS[i] for i in top3]

    scores.append(
        apk(row["answer"], preds)
    )

map3 = np.mean(scores)

print("MAP@3:", round(map3, 4))

MAP@3: 0.3267
